# Volcano Data And Sql

Volcanic event data from a public NOAA API, salary queries using SQLite, and pandas aggregation. Cells are preserved from my University of Maryland coursework; plots and output snapshots have been removed for a cleaner presentation.


## Volcano data: API extraction and preparation


In [ ]:
# Make sure these code blocks run properly and that you have properly installed the appropriate modules required.
import pandas as pd
import requests
# import other libraries here

# Don't Remove this
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [ ]:
# API URL and headers in case request gets denied.
api_url = "https://www.ngdc.noaa.gov/hazel/hazard-service/api/v1/volcanoes"

headers = {
    'accept': '*/*'
}

In [ ]:
response = requests.get(api_url, headers = headers)
data = response.json()
df = pd.DataFrame(data['items'])
df.to_csv('volcanoes.csv', index = False)

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
new_df = df[['id', 'year', 'month', 'day', 'tsunamiEventId', 'earthquakeEventId', 'volcanoLocationId', 
             'volcanoLocationNewNum', 'name', 'country', 'elevation', 'morphology', 'deathsTotal', 'vei', 'deaths']]

In [ ]:
new_df = new_df.dropna(subset = ['year', 'month', 'day'])

In [ ]:
new_df.index = range(1, len(new_df) + 1)

In [ ]:
new_df['totalDeaths'] = new_df[['deathsTotal', 'deaths']].max(axis = 1)


In [ ]:
new_df[['year', 'month', 'day']] = new_df[['year', 'month', 'day']].astype(int)

new_df['date'] = pd.to_datetime(new_df[['year', 'month', 'day']], errors = 'coerce')

# Drop original columns and sort.
new_df = new_df.drop(columns= ['year', 'month', 'day'])
new_df = new_df.sort_values(by= 'date', ascending= False)

# Place new column next to 'id' column.
date = new_df.pop('date')
new_df.insert(1, 'date', date)

# KEEP THIS. It will display the whole dataframe.
new_df

## Volcanic event exploration


In [ ]:
from IPython.display import display
for country, c_df in new_df.groupby('country'):
    c_df = c_df[['date', 'country', 'name', 'vei']]
    c_df = c_df.sort_values(by= 'vei', ascending= False)
    display(c_df)


In [ ]:
c_maxvei = new_df.groupby('country')['vei'].max()

for country, vei in c_maxvei.items():
    print(f"Country: {country}, Highest VEI: {vei}")

In [ ]:
import matplotlib.pyplot as plt

v_counts = new_df.groupby('name').size()
v_names = v_counts[v_counts > 4].index

for v in v_names:
    v_df = new_df[new_df['name'] == v]

    plt.figure()
    plt.plot(v_df['date'].dt.year, v_df['vei'], marker = 'o', label = v)
    plt.title(v)
    plt.xlabel('Year')
    plt.ylabel('VEI')
    plt.legend()
    plt.show()
    
# not adding the graphs here, but the graphs should have the volcano name as the title
# the x axis should be the years and the y axis should be the VEI
# A graph that doesn't have these things will only get partial points



## SQL analysis of salary records


In [ ]:
import sqlite3
import pandas as pd # Pandas was already imported from the previous sections

conn = sqlite3.connect("database.sqlite")
crsr = conn.cursor()

In [ ]:
# This code will let you check out the different tables within the database.
query = "SELECT name FROM sqlite_master WHERE type='table';"
tables = crsr.execute(query).fetchall()
print(tables)

In [ ]:
query = '''
SELECT AVG(TotalPay)
FROM Salaries
WHERE Year BETWEEN 2011 AND 2014
AND JobTitle LIKE '%clerk%'

'''

# KEEP THIS. It will display the whole dataframe.
df = pd.read_sql(query, conn)
df

In [ ]:
query = '''
SELECT JobTitle, TotalPay
FROM Salaries
WHERE Year = 2013
AND JobTitle LIKE '%clerk'
AND TotalPay < 50000
ORDER BY TotalPay DESC


'''
# KEEP THIS. It will display the whole dataframe.
df = pd.read_sql(query, conn)
df

In [ ]:
query = '''

SELECT AVG(BasePay) AS avg_base_pay,
       AVG(Benefits) AS avg_benefits,
       AVG(OvertimePay) AS avg_overtime_pay,
       AVG(BasePay) + AVG(Benefits) + AVG(OvertimePay) AS total_avg_compensation
FROM Salaries
WHERE JobTitle LIKE '%clerk%'
AND JobTitle NOT GLOB '*SENIOR*'
'''

# KEEP THIS. It will display the whole dataframe.
df = pd.read_sql(query, conn)
df

In [ ]:
for year in range(2010, 2016):
    query = f'''
    SELECT *
    FROM Salaries
    WHERE Year = {year};
    '''
    
    year_df = pd.read_sql(query, conn)
    
    year_df.to_sql(f'Salaries_{year}',conn, if_exists='replace', index= False)

In [ ]:
# Run this code to check if you successfully added your table.
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print(cursor.fetchall())

## Grouping and order aggregation


In [ ]:
# Create the Sample dataset from above flowchart
input_data = {
    'account': ['383080', '383080', '383080', '412290', '412290','412290','412290','412290','218895','218895','218895','218895'],
    'order': ['10001', '10001', '10001', '10005', '10005','10005','10005','10005','10006','10006','10006','10006'],
    'ext_price': [235.83, 232.32, 107.97, 2679.36, 286.02, 832.95, 3472.04, 915.12, 3061.12, 518.65, 216.9, -72.18]
}

# Create DataFrame
df = pd.DataFrame(input_data)

# Display Dataframe (DONT REMOVE THE CODE)
df

In [ ]:
# Group by 'order' and show intermediate result
g = df.groupby('order')

# Display intermediate result for each group; hints: you have to use 'for loop'
for order, group in g:
    print("Order: ", order)
    display(group)



In [ ]:

# Repeat group by 'order' again and then apply aggregation (sum of 'ext_price' for each 'order')
g_sum = df.groupby('order')['ext_price'].sum()

# Show the aggregated result after re-grouping  (DONT REMOVE THE CODE)
g_sum



In [ ]:
# Reset index to combine result into a single DataFrame
final = g_sum.reset_index()
# Rename the columns for clarity
final.columns = ['order', 'total_ext_price']

# Show the final result  (DONT REMOVE THE CODE)
final